In [6]:
import requests
import json
import time

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.7, max_tokens=300):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "think": False,
                "options": {
                    "temperature": temperature,
                    "num_predict": max_tokens
                }
            },
            timeout=60
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

print("Ollama LLM helper loaded")

Ollama LLM helper loaded


In [7]:
with open("../outputs/resume_text.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

with open("../outputs/skill_gap.json", "r", encoding="utf-8") as f:
    gap_data = json.load(f)

with open("../outputs/career_readiness.json", "r", encoding="utf-8") as f:
    readiness_data = json.load(f)

missing_skills = gap_data["missing_skills"]
ats_feedback = readiness_data["ats"]["feedback"]

print("Loaded resume text, skill gap, and ATS feedback")
print(f"ATS feedback: {ats_feedback}")

Loaded resume text, skill gap, and ATS feedback
ATS feedback: ['Great! Your resume covers the key ATS-friendly elements']


In [8]:
import re

def suggest_resume_improvements(text, missing_skills, ats_feedback):
    suggestions = []

    if missing_skills:
        suggestions.append({
            "category": "Missing Keywords",
            "suggestion": f"Consider adding these in-demand skills if you have experience with them: {', '.join(missing_skills[:5])}"
        })

    text_upper = text.upper()
    required_sections = ["EXPERIENCE", "CERTIFICATION", "ACHIEVEMENT"]
    for section in required_sections:
        if section not in text_upper:
            suggestions.append({
                "category": "Missing Section",
                "suggestion": f"Consider adding a '{section.title()}' section to strengthen your resume"
            })

    has_linkedin = bool(re.search(r'linkedin\.com', text, re.IGNORECASE))
    has_github = bool(re.search(r'github\.com', text, re.IGNORECASE))

    if not has_linkedin:
        suggestions.append({
            "category": "Contact Info",
            "suggestion": "Add your LinkedIn profile URL for recruiters to find you easily"
        })
    if not has_github:
        suggestions.append({
            "category": "Contact Info",
            "suggestion": "Add your GitHub profile to showcase your projects and code quality"
        })

    project_section_has_numbers = bool(re.search(r'\d+%|\d+x|\d+ (users|projects|hours)', text))
    if not project_section_has_numbers:
        suggestions.append({
            "category": "Impact Metrics",
            "suggestion": "Add measurable impact to your projects (e.g., 'improved accuracy by 15%', 'reduced processing time by 2x')"
        })

    for fb in ats_feedback:
        if fb != "Great! Your resume covers the key ATS-friendly elements":
            suggestions.append({"category": "ATS Optimization", "suggestion": fb})

    return suggestions

improvement_suggestions = suggest_resume_improvements(resume_text, missing_skills, ats_feedback)

print("Rule-Based Improvement Agent: SUCCESS\n")
for i, s in enumerate(improvement_suggestions, 1):
    print(f"{i}. [{s['category']}] {s['suggestion']}")

Rule-Based Improvement Agent: SUCCESS

1. [Missing Keywords] Consider adding these in-demand skills if you have experience with them: deep learning, docker, fastapi, kubernetes, langchain
2. [Missing Section] Consider adding a 'Achievement' section to strengthen your resume
3. [Contact Info] Add your LinkedIn profile URL for recruiters to find you easily
4. [Contact Info] Add your GitHub profile to showcase your projects and code quality


In [9]:
def get_llm_resume_feedback(resume_text, missing_skills):
    prompt = f"""You are an expert technical resume reviewer for AI/ML roles.

Analyze this resume and give 4-5 specific, actionable improvement suggestions.
Focus on: content gaps, impact/quantification, clarity, and relevance to AI Engineering roles.
Do NOT suggest generic advice like "use action verbs" - be specific to THIS resume.

Missing skills the candidate should consider highlighting if they have experience: {', '.join(missing_skills[:5])}

RESUME TEXT:
{resume_text[:2000]}

Respond as a numbered list of 4-5 suggestions, each 1-2 sentences. No preamble, just the list."""

    return call_llm(prompt, temperature=0.5, max_tokens=400)

start = time.time()
llm_feedback = get_llm_resume_feedback(resume_text, missing_skills)
elapsed = time.time() - start

print(f"LLM Resume Feedback Agent: SUCCESS (generated in {elapsed:.2f}s)\n")
print(llm_feedback)

LLM Resume Feedback Agent: SUCCESS (generated in 6.90s)

Replace "Digital Marketing" in your skills section with **Deep Learning** or **PyTorch**, given you already listed TensorFlow and PyTorch; this directly addresses a critical gap for AI Engineering roles while leveraging existing coursework. Add **Docker** and **FastAPI** to your skill list if you have used them, even briefly during projects like the supply chain system, as these are standard tools expected by modern ML engineers for deployment and serving models. Reformat your project descriptions using the STAR method with specific metrics (e.g., "reduced prediction error by X%" or "processed Y records") instead of generic phrases like "analyze data" to quantify your actual impact on business problems. Replace **Leadership** in your skills section with relevant technical competencies such as **Git**, **Linux**, or **CI/CD pipelines**, since leadership is a soft skill that does not demonstrate the engineering capabilities recruit

In [10]:
combined_improvements = {
    "rule_based_suggestions": improvement_suggestions,
    "llm_generated_feedback": llm_feedback,
    "llm_generation_time_seconds": round(elapsed, 2)
}

with open("../outputs/resume_improvements.json", "w", encoding="utf-8") as f:
    json.dump(combined_improvements, f, indent=2)

print(f"\n{len(improvement_suggestions)} rule-based suggestions + LLM analysis saved to ../outputs/resume_improvements.json")
print("Notebook 8 (Resume Improvement Agent) — UPGRADED WITH LLM — COMPLETE")


4 rule-based suggestions + LLM analysis saved to ../outputs/resume_improvements.json
Notebook 8 (Resume Improvement Agent) — UPGRADED WITH LLM — COMPLETE
